In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import re

In [2]:
# Step 1: Fetch the page content
url = "https://en.wikipedia.org/wiki/List_of_cinematic_firsts"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

output_file = 'cinema_timeline.csv'

# Focus on the main content section (the timeline events are within mw-content-text)
content_div = soup.find('div', {'id': 'mw-content-text'})

# Open the output CSV file to write the cleaned data
with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Date', 'Event Description'])

    events = []
    current_year = None
    start_processing = False

    # Begin parsing within the content div
    for section in content_div.find_all(['h2', 'h3', 'p', 'ul']):
        # Stop parsing if we reach sections like "See also" or "References"
        if 'See also' in section.text or 'Further reading' in section.text or 'References' in section.text or 'Bibliography' in section.text:
            break

        # Extract potential decade or specific year from headings (like <h2>, <h3>, etc.)
        if section.name in ['h2', 'h3']:
            heading_text = section.text.strip()

            # Skip irrelevant century and decade headings
            if 'century' in heading_text.lower() or 'decade' in heading_text.lower():
                continue

            # If it's a valid year, assign it as current_year and start processing
            match = re.search(r'(\d{4})', heading_text)
            if match:
                current_year = match.group(1)
                start_processing = True
                print(f"Processing year: {current_year}")

        # Only process sections after encountering the first valid year
        if start_processing and section.name == 'ul':  # Events are often listed in bullet points
            for li in section.find_all('li'):
                event_text = li.text.strip()
                if ": " in event_text:
                    # Split to remove month/day, keep only event description
                    #event_description = event_text.split(": ", 1)[1]
                    # Remove double quotes
                    event_descriptio = re.sub(r'"', '', event_description).strip()
                    # Remove reference notes (e.g., [1])
                    event_description = re.sub(r'\[\d+\]', '', event_description).strip()
                    events.append((current_year, event_description))

    # Write the events to the CSV file
    writer.writerows(events)

print("CSV file created with cinema events.")

CSV file created with cinema events.


In [3]:
import requests
from bs4 import BeautifulSoup
import csv
import re

# Step 1: Fetch the page content
url = "https://en.wikipedia.org/wiki/List_of_cinematic_firsts"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

output_file = 'prova_cinema.csv'

# Focus on the main content section (the timeline events are within mw-content-text)
content_div = soup.find('div', {'id': 'mw-content-text'})

# Open the output CSV file to write the cleaned data
with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['Date', 'Event Description'])

    events = []
    current_year = None

    # Begin parsing within the content div
    for section in content_div.find_all(['h2', 'h3', 'ul']):
        # Stop parsing if we reach sections like "See also" or "References"
        if 'See also' in section.text or 'Further reading' in section.text or 'References' in section.text or 'Bibliography' in section.text:
            break

        # Extract specific year from headings (like <h2>, <h3>, etc.)
        if section.name in ['h2', 'h3']:
            heading_text = section.text.strip()

            # Skip decade headings by checking if it contains "s" (e.g., "1870s")
            if 'century' in heading_text.lower() or 'decade' in heading_text.lower() or re.search(r'\b\d{3}0s\b', heading_text) or re.search('\b\d{4}th\b', heading_text) or re.search(r'pre-\d{4}', heading_text):
                current_year = None  # Reset current_year if it's a decade heading
                continue

            # If it's a valid year, set it as current_year
            match = re.search(r'\b(\d{4})\b', heading_text)
            if match:
                current_year = match.group(1)
                print(f"Processing year: {current_year}")

        # Only process <ul> sections after encountering a valid year
        if current_year and section.name == 'ul':  # Events are often listed in bullet points
            for li in section.find_all('li'):
                event_text = li.text.strip()

                # Remove reference notes (e.g., [1]) and any other square-bracketed text
                event_description = re.sub(r'\[\d+\]', '', event_text).strip()

                # Append the event with the current year
                events.append((current_year, event_description))

    # Write the events to the CSV file
    writer.writerows(events)

print("CSV file created with cinema events.")



Processing year: 1870
CSV file created with cinema events.


In [ ]:
####### To check the format of the wikipedia page content #######

# Step 1: Fetch the page content
url = "https://en.wikipedia.org/wiki/List_of_cinematic_firsts"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

output_file = 'prova_cinema.csv'

# Focus on the main content section (the timeline events are within mw-content-text)
content_div = soup.find('div', {'id': 'mw-content-text'})

# Check if the content_div is found
if content_div:
    print("Content div found.")
else:
    print("Content div not found.")

# Open the output CSV file to write the cleaned data
with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['Year', 'Event Description'])

    events = []
    current_year = None
    timeline_started = False  # Flag to skip introduction

    # Begin parsing within the content div
    for section in content_div.find_all(['h2', 'h3', 'p', 'ul'], recursive=True):  # Set to True to capture all nested tags
        print(f"Inspecting section: {section.name} - {section.text[:50]}")  # Log each section briefly for inspection

        # Skip introductory text until we detect a valid timeline section
        if not timeline_started:
            if section.name in ['h2', 'h3'] and re.search(r'\d{4}', section.text):
                timeline_started = True  # Start parsing from the first valid date heading
                print("Timeline section found. Starting to parse events.")
            else:
                continue  # Skip sections until the timeline starts

        # Stop parsing if we reach sections like "See also" or "References"
        if 'See also' in section.text or 'Further reading' in section.text or 'References' in section.text or 'Bibliography' in section.text:
            print("Reached end of relevant sections.")
            break

        # Extract specific year from headings (like <h2>, <h3>, etc.)
        if section.name in ['h2', 'h3']:
            heading_text = section.text.strip()
            print(f"Found heading: {heading_text}")

            # Skip non-specific titles (centuries or decades) using regex
            if re.search(r'\b(?:\d{2}(?:00s|0s|[1-9]th century)|century|decade|pre-\d{4})\b', heading_text, re.IGNORECASE):
                current_year = None  # Reset current_year if it's a non-specific heading
                print("Skipped non-specific title.")
                continue

            # If it's a valid precise year, set it as current_year
            match = re.search(r'\b(\d{4})\b', heading_text)
            if match:
                current_year = match.group(1)
                print(f"Processing year: {current_year}")

        # Process any content (e.g., paragraphs or lists) following a valid year heading
        if current_year:
            # Capture direct list items or paragraphs within the section
            if section.name in ['ul', 'p']:
                items = section.find_all(['li', 'p']) if section.name == 'ul' else [section]

                for item in items:
                    event_text = item.text.strip()
                    if event_text:  # Ensure there's content to process
                        print(f"Found event for year {current_year}: {event_text}")

                        # Remove reference notes (e.g., [1]) and any other square-bracketed text
                        event_description = re.sub(r'\[\d+\]', '', event_text).strip()

                        # Append the event with the current year
                        events.append((current_year, event_description))

            # Check sibling nodes after each valid year
            sibling = section.find_next_sibling()
            while sibling and sibling.name not in ['h2', 'h3']:
                if sibling.name in ['ul', 'p']:
                    items = sibling.find_all(['li', 'p']) if sibling.name == 'ul' else [sibling]

                    for item in items:
                        event_text = item.text.strip()
                        if event_text:
                            print(f"Found sibling event for year {current_year}: {event_text}")
                            event_description = re.sub(r'\[\d+\]', '', event_text).strip()
                            events.append((current_year, event_description))

                sibling = sibling.find_next_sibling()

    # Check if events were captured
    if events:
        print("Events captured:", events)
    else:
        print("No events captured.")

    # Write the events to the CSV file
    writer.writerows(events)

print("CSV file created with cinema events.")



Content div found.
Inspecting section: p - This page lists chronologically the first achievem
Inspecting section: p - In parallel with the developments in technology, i
Inspecting section: p - 19th century: Pre-1870 • 1870s • 1880s • 1890s
20t
Inspecting section: h2 - 19th century
Inspecting section: h3 - Pre-1870
Timeline section found. Starting to parse events.
Found heading: Pre-1870
Skipped non-specific title.
Inspecting section: ul - Peter Mark Roget's wrote the article Explanation o
Inspecting section: ul - Almost simultaneously, around December 1832, the B
Inspecting section: h3 - 1870s
Found heading: 1870s
Inspecting section: ul - French astronomer P.J.C. Janssen came up with the 
Inspecting section: ul - Using a battery of 12 cameras Eadweard Muybridge r
Inspecting section: h3 - 1880s
Found heading: 1880s
Inspecting section: ul - During his lectures on locomotion, Eadweard Muybri
Inspecting section: ul - Étienne-Jules Marey developed the Chronophotograph
Inspecting section: ul

In [19]:
import requests
from bs4 import BeautifulSoup
import csv
import re

# Step 1: Fetch the page content
url = "https://en.wikipedia.org/wiki/List_of_cinematic_firsts"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

output_file = 'cinema_events.csv'

# Focus on the main content section (the timeline events are within mw-content-text)
content_div = soup.find('div', {'id': 'mw-content-text'})

# Check if the content_div is found
if content_div:
    print("Content div found.")
else:
    print("Content div not found.")

# Open the output CSV file to write the cleaned data
with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['Year', 'Event Description'])

    events = []
    timeline_started = False  # Flag to skip introduction

    # Begin parsing within the content div
    for section in content_div.find_all(['h2', 'h3', 'ul'], recursive=True):  # Process h2, h3, and ul tags
        print(f"Inspecting section: {section.name} - {section.text[:50]}")  # Log each section briefly for inspection

        # Skip introductory text until we detect a valid timeline section
        if not timeline_started:
            if section.name in ['h2', 'h3'] and re.search(r'\d{4}', section.text):
                timeline_started = True  # Start parsing from the first valid date heading
                print("Timeline section found. Starting to parse events.")
            else:
                continue  # Skip sections until the timeline starts

        # Stop parsing if we reach sections like "See also" or "References"
        if 'See also' in section.text or 'Further reading' in section.text or 'References' in section.text or 'Bibliography' in section.text:
            print("Reached end of relevant sections.")
            break

        # Process list items in <ul> tags for specific years and events
        if section.name == 'ul':
            for li in section.find_all('li'):
                event_text = li.text.strip()

                # Search for specific years within the list item
                year_matches = re.findall(r'\b(18\d{2}|19\d{2}|20\d{2})\b', event_text)
                if year_matches:
                    # Take the first matching year as the primary year for this event
                    for year in year_matches:
                        # Remove reference notes (e.g., [1]) and any other square-bracketed text
                        event_description = re.sub(r'\[\d+\]', '', event_text).strip()

                        # Ensure that each event is unique
                        if (year, event_description) not in events:
                            print(f"Found event for year {year}: {event_description}")
                            events.append((year, event_description))
                        else:
                            print(f"Duplicate event for year {year} ignored.")

    # Check if events were captured
    if events:
        print("Events captured:", events)
    else:
        print("No events captured.")

    # Write the events to the CSV file
    writer.writerows(events)

print("CSV file created with cinema events.")



Content div found.
Inspecting section: h2 - 19th century
Inspecting section: h3 - Pre-1870
Timeline section found. Starting to parse events.
Inspecting section: ul - Peter Mark Roget's wrote the article Explanation o
Inspecting section: ul - Almost simultaneously, around December 1832, the B
Found event for year 1832: Almost simultaneously, around December 1832, the Belgian physicist Joseph Plateau and the Austrian professor of practical geometry Simon Stampfer invented the Phenakistiscope, the first practical device to create a fluid illusion of motion. Plateau introduced the device in January 1833 in a scientific magazine.
Found event for year 1833: Almost simultaneously, around December 1832, the Belgian physicist Joseph Plateau and the Austrian professor of practical geometry Simon Stampfer invented the Phenakistiscope, the first practical device to create a fluid illusion of motion. Plateau introduced the device in January 1833 in a scientific magazine.
Inspecting section: h3 - 18